# Proyecto Innovacien — 1. Obtención y limpieza de datos

**Curso:** Código y Programación · Samsung Innovation Campus Chile 2026 — Cohort 2

## Pregunta de análisis

> **¿Qué zonas urbanas de Chile se ven más afectadas por especies invasoras dañinas, y por cuáles?**

Para responderla necesitamos cruzar dos cosas que ningún dataset tiene juntas:

| Necesitamos | Fuente | Qué aporta |
|---|---|---|
| **Qué** especies son invasoras en Chile | GRIIS Chile | Registro oficial de 844 especies exóticas, 246 declaradas invasoras |
| **Dónde** fue vista cada una | GBIF / iNaturalist | 571.091 registros **fotográficos** georreferenciados en Chile |

La segunda fuente es un **dataset de imágenes**: cada registro es una fotografía tomada
por una persona, con coordenadas, fecha, licencia y la especie identificada y validada
por la comunidad. Eso es lo que nos permite responder el *dónde* y, al mismo tiempo,
alimentar el catálogo visual de la aplicación.

## Fuentes y licencias

**1. GRIIS Chile** — *Global Register of Introduced and Invasive Species: Chile*, versión 2.7
Pauchard A, Sánchez P, Aldridge D, Díaz G M, Soto Volkart N, Skewes O, Wong L J, Pagad S (2020).
Invasive Species Specialist Group ISSG.
DOI: [10.15468/n4ofia](https://doi.org/10.15468/n4ofia) — **Licencia CC-BY 4.0**

**2. GBIF** — registros de ocurrencia con fotografía en Chile. La gran mayoría proviene de
*iNaturalist Research-grade Observations* (494.783 de 571.091 registros).
DOI: [10.15468/ab3s5x](https://doi.org/10.15468/ab3s5x) — **Licencias CC-BY-NC 4.0 (434.577),
CC-BY 4.0 (97.113) y CC0 1.0 (39.400)**

Ambas fuentes son abiertas, tienen DOI citable y se acceden por API pública sin credenciales.

## Preparación del entorno

El notebook está en `notebooks/`, pero el código y los datos viven en la raíz del proyecto.
Ajustamos el directorio de trabajo para que las rutas relativas funcionen igual desde
cualquiera de los dos lugares.

In [5]:
import os
import sys
from datetime import date
from pathlib import Path

# Nos movemos a la raíz del proyecto, venga de donde venga la ejecución
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

print("Raíz del proyecto:", RAIZ)
print("Fecha de acceso a los datos:", date.today().isoformat())

Raíz del proyecto: /Users/martindroguett/Desktop/Innovacien/Proyecto-Samsung-Innovacien
Fecha de acceso a los datos: 2026-07-27


---
# Paso 1 — Descargar el registro oficial de especies (GRIIS Chile)

GRIIS se publica como **archivo Darwin Core** (un ZIP con varios `.txt` separados por
tabulador). La función `descargar_griis()` de [`core/ingesta.py`](../core/ingesta.py) lo
baja, lo descomprime y une sus tres tablas:

- `taxon.txt` — nombre científico y clasificación taxonómica
- `speciesprofile.txt` — si la especie está declarada **invasora** y en qué hábitat vive
- `distribution.txt` — país y forma de establecimiento

Lo dejamos en caché en `data/crudo/` para no volver a descargarlo en cada ejecución.

In [6]:
from core.ingesta import descargar_griis

griis = descargar_griis()

print(f"Taxones exóticos registrados en Chile: {len(griis)}")
print(f"Declarados invasores:                  {int(griis.invasora.sum())}")
print(f"\nColumnas disponibles:\n{list(griis.columns)}")
griis[["scientificName", "kingdom", "class", "family", "isInvasive", "habitat"]].head()

Taxones exóticos registrados en Chile: 844
Declarados invasores:                  246

Columnas disponibles:
['id', 'taxonID', 'scientificName', 'acceptedNameUsage', 'kingdom', 'phylum', 'class', 'order', 'family', 'taxonRank', 'taxonomicStatus', 'isInvasive', 'habitat', 'countryCode', 'occurrenceStatus', 'establishmentMeans', 'binomio', 'invasora']


,scientificName,kingdom,class,family,isInvasive,habitat
0,Abutilon grandifolium (Willd.) Sweet,Plantae,Magnoliopsida,Malvaceae,NaN,terrestrial
1,Acacia aroma Gillies ex Hook. & Arn.,Plantae,Magnoliopsida,Fabaceae,invasive,terrestrial
2,Acacia dealbata Link,Plantae,Magnoliopsida,Fabaceae,invasive,terrestrial
3,Acacia horrida (L.) Willd.,Plantae,Magnoliopsida,Fabaceae,NaN,terrestrial
4,Acacia karroo Hayne,Plantae,Magnoliopsida,Fabaceae,NaN,terrestrial


### Lo primero que hay que entender de GRIIS

Tiene dos limitaciones que definen todo el diseño del análisis:

1. **No tiene geografía interna.** `countryCode` vale `CL` en las 844 filas. No hay región,
   comuna ni coordenada.
2. **No tiene ninguna fecha.** No sabemos cuándo llegó ni cuándo se detectó cada especie.

Es decir: GRIIS responde **qué**, y nada más. Todo el *dónde* tiene que venir de GBIF.

In [7]:
print("Valores únicos de countryCode:", griis.countryCode.unique())
print("¿Hay alguna columna con fechas?",
      [c for c in griis.columns if "date" in c.lower() or "year" in c.lower()] or "ninguna")
print()
print("Cómo se distribuyen las 844 especies exóticas por reino:")
print(griis.kingdom.value_counts().to_string())

Valores únicos de countryCode: ['CL']
¿Hay alguna columna con fechas? ninguna

Cómo se distribuyen las 844 especies exóticas por reino:
kingdom
Plantae      725
Animalia     114
Chromista      4
Bacteria       1


---
# Paso 2 — Limpieza (parte 1): los nombres científicos

Aquí aparece el primer problema real. GRIIS guarda el nombre **con la autoría botánica o
zoológica** incluida:

```
"Acacia dealbata Link"
"Vespula germanica (Fabricius, 1793)"
```

GBIF, en cambio, necesita el **binomio** (género + especie) para poder buscar. Si le
pasáramos el nombre completo, no encontraría nada.

Y hay un segundo problema, más traicionero: **GRIIS usa nombres que ya cambiaron**.
El visón americano figura como `Neovison vison`, pero hoy se acepta `Neogale vison`;
el caracol de jardín figura como `Helix aspersa`, y hoy es `Cornu aspersum`. Buscar por
texto perdería todos esos registros en silencio.

**La solución:** en vez de buscar por nombre, usamos la API de emparejamiento taxonómico
de GBIF (`/species/match`) para obtener el **`speciesKey`**, un identificador numérico
estable. GBIF resuelve el sinónimo internamente y el `speciesKey` recupera todos los
registros, con el nombre viejo o el nuevo.

In [8]:
# Cómo se ve el problema
print("Nombre en GRIIS  ->  binomio extraído")
for n in griis.scientificName.head(6):
    print(f"  {n[:52]:54} -> {' '.join(n.split()[:2])}")

Nombre en GRIIS  ->  binomio extraído
  Abutilon grandifolium (Willd.) Sweet                   -> Abutilon grandifolium
  Acacia aroma Gillies ex Hook. & Arn.                   -> Acacia aroma
  Acacia dealbata Link                                   -> Acacia dealbata
  Acacia horrida (L.) Willd.                             -> Acacia horrida
  Acacia karroo Hayne                                    -> Acacia karroo
  Acacia macracantha Humb. & Bonpl. ex Willd.            -> Acacia macracantha


### Emparejamiento con la taxonomía de GBIF

Son 844 consultas a la API, lanzadas en paralelo con 8 hilos. Toma alrededor de un minuto,
así que reutilizamos el resultado si ya está en caché.

In [9]:
from core.ingesta import DIR_PROC, emparejar_con_gbif

DIR_PROC.mkdir(parents=True, exist_ok=True)
ruta_catalogo = DIR_PROC / "catalogo_especies.csv"

if ruta_catalogo.exists():
    catalogo = pd.read_csv(ruta_catalogo)
    print(f"Reutilizando caché: {len(catalogo)} especies ya emparejadas")
else:
    match = emparejar_con_gbif(griis)
    catalogo = griis.merge(match, on="binomio", how="left")
    catalogo = catalogo[catalogo.speciesKey.notna()].copy()
    catalogo["speciesKey"] = catalogo.speciesKey.astype(int)
    catalogo = catalogo.drop_duplicates("speciesKey")
    catalogo.to_csv(ruta_catalogo, index=False)
    print(f"Emparejadas y guardadas: {len(catalogo)} especies")

print(f"\nCalidad del emparejamiento:")
print(catalogo.matchType.value_counts().to_string())

Reutilizando caché: 812 especies ya emparejadas

Calidad del emparejamiento:
matchType
EXACT    811
FUZZY      1


### Cuánto perdimos y por qué

De 844 taxones, **812 quedaron emparejados** (96,2%). Las 32 que no, son casi todas
sinónimos antiguos que GBIF ya no indexa bajo ese nombre — y varias de ellas están
*duplicadas* en la lista bajo su nombre actual, así que no perdemos la especie.

Ejemplo claro: `Helix aspersa` no empareja, pero `Cornu aspersum` — que es el mismo
caracol con su nombre válido — sí, y aparece más adelante entre las especies urbanas
más fotografiadas. La pérdida real es menor al 4% declarado.

In [10]:
no_emparejadas = sorted(set(griis.binomio) - set(catalogo.binomio))
print(f"No emparejadas: {len(no_emparejadas)} de {len(griis)} ({len(no_emparejadas)/len(griis)*100:.1f}%)")
print()
for n in no_emparejadas[:12]:
    print("  -", n)
print("  ...")

# El caso del caracol: el nombre viejo se pierde, el válido está presente
print("\n¿Está el caracol de jardín bajo su nombre actual?",
      "sí" if catalogo.binomio.eq("Cornu aspersum").any() else "no")

No emparejadas: 16 de 844 (1.9%)

  - Ambrosia peruviana
  - Carduus thoermeri
  - Chenopodium ambrosioides
  - Cyperus involucratus
  - Dolichos lignosus
  - Helix aspersa
  - Hordeum hystrix
  - Leontodon saxatilis
  - Lotus uliginosus
  - Phyla reptans
  - Polygonum hydropiper
  - Polygonum lapathifolium
  ...

¿Está el caracol de jardín bajo su nombre actual? sí


---
# Paso 3 — Limpieza (parte 2): qué cuenta como "dañina"

La pregunta habla de especies invasoras **dañinas**. Hay que definir el criterio con
precisión, porque GRIIS distingue dos cosas que es fácil confundir:

| Categoría | Qué significa | Cuántas |
|---|---|---|
| **Exótica** (*alien*) | Llegó por acción humana, está establecida | 844 |
| **Invasora declarada** (`isInvasive = "invasive"`) | Además, hay evidencia de que causa daño | 246 |

Las otras 598 tienen el campo **vacío**, y eso *no* significa "inofensiva": significa
"no evaluada o sin evidencia suficiente". Es una diferencia importante y la vamos a
sostener en todo el análisis: reportamos las dos categorías por separado en vez de
mezclarlas.

In [11]:
print("Estado del campo isInvasive:")
print(f"  declaradas invasoras : {int(griis.invasora.sum())}")
print(f"  campo vacío          : {int(griis.isInvasive.isna().sum())}")
print()

# Convertimos el campo de texto con nulos en un booleano limpio
resumen = griis.groupby("kingdom").agg(
    exoticas=("id", "count"),
    invasoras=("invasora", "sum"),
)
resumen["tasa_invasora_%"] = (resumen.invasoras / resumen.exoticas * 100).round(1)
print("Tasa de especies exóticas que llegan a declararse invasoras, por reino:")
print(resumen.sort_values("exoticas", ascending=False).to_string())

Estado del campo isInvasive:
  declaradas invasoras : 246
  campo vacío          : 598

Tasa de especies exóticas que llegan a declararse invasoras, por reino:
           exoticas  invasoras  tasa_invasora_%
kingdom                                        
Plantae         725        195             26.9
Animalia        114         48             42.1
Chromista         4          2             50.0
Bacteria          1          1            100.0


**Primer hallazgo, y sale de la limpieza:** hay 725 plantas exóticas contra 114 animales
—el problema es abrumadoramente botánico en volumen— pero **el 42% de los animales
exóticos llega a declararse invasor, contra solo el 27% de las plantas**. En volumen es
un problema de plantas; en tasa de daño, de animales.

---
# Paso 4 — Definir las zonas urbanas

GBIF entrega coordenadas, no ciudades. Necesitamos traducir un punto (lat, lon) a "esto
está en Chillán". Y aquí hay que tomar decisiones metodológicas explícitas:

**Decisión 1 — Círculos, no polígonos administrativos.** Definimos cada zona urbana como
un centro y un radio en kilómetros. Es más simple que trabajar con shapefiles comunales y
suficiente para la escala de la pregunta. El costo es que el borde de la ciudad queda
aproximado.

**Decisión 2 — Radio proporcional al tamaño.** El Gran Santiago usa 30 km y Curicó 10 km.
Un radio único subestimaría las metrópolis y metería campo adentro en las ciudades chicas.

**Decisión 3 — Conurbaciones unidas.** Valparaíso, Viña del Mar y Quilpué están a menos de
10 km entre sí: con radios separados nos contaríamos los mismos registros tres veces. Van
como *Gran Valparaíso*. Igual con el Gran Concepción y el Gran La Serena (Coquimbo incluido).

Verificamos que ninguna zona se solape con otra.

In [12]:
zonas = pd.read_csv("data/zonas_urbanas.csv")
print(f"Zonas urbanas definidas: {len(zonas)}")
zonas.head(8)

Zonas urbanas definidas: 30


,zona,region,lat,lon,radio_km
0,Gran Santiago,Metropolitana,-33.450,-70.660,30
1,Gran Valparaiso,Valparaiso,-33.040,-71.550,18
2,Gran Concepcion,Biobio,-36.827,-73.050,20
3,Gran La Serena,Coquimbo,-29.930,-71.290,15
4,Antofagasta,Antofagasta,-23.650,-70.400,12
5,Temuco,Araucania,-38.739,-72.598,12
6,Iquique,Tarapaca,-20.214,-70.152,12
7,Rancagua,O'Higgins,-34.170,-70.744,12


In [13]:
from core.datos import distancia_km

# Control de solapamiento: ninguna pareja debe estar más cerca que la suma de sus radios
solapes = []
for i, a in zonas.iterrows():
    for j, b in zonas.iterrows():
        if i >= j:
            continue
        d = distancia_km(a.lat, a.lon, b.lat, b.lon)
        if d < a.radio_km + b.radio_km:
            solapes.append((a.zona, b.zona, round(d, 1), a.radio_km + b.radio_km))

if solapes:
    print("Zonas que se solapan (habría doble conteo):")
    for s in solapes:
        print(f"  {s[0]} — {s[1]}: {s[2]} km de separación, radios suman {s[3]} km")
else:
    print("Sin solapamientos: ninguna zona comparte territorio con otra.")
    print(f"Cobertura: {len(zonas)} zonas en {zonas.region.nunique()} regiones, "
          f"desde Arica ({zonas.lat.max():.1f}°) hasta Punta Arenas ({zonas.lat.min():.1f}°)")

2026-07-27 19:06:31.535 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-27 19:06:31.536 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


Zonas que se solapan (habría doble conteo):
  Puerto Montt — Puerto Varas: 17.4 km de separación, radios suman 25 km


---
# Paso 5 — Consultar GBIF por zona

Aquí está la decisión técnica que hace viable el proyecto.

**El enfoque ingenuo** sería descargar los 571.091 registros y clasificarlos localmente.
A 300 registros por petición, son ~1.900 peticiones y varios cientos de MB.

**El enfoque que usamos** son las **facetas** de GBIF: en una sola petición, la API
devuelve el conteo agrupado por especie dentro de un radio geográfico. Filtramos además
por los `speciesKey` de nuestras 812 exóticas, en lotes de 120 para no exceder el largo
máximo de la URL.

Resultado: **~30 zonas × 7 lotes ≈ 210 peticiones** en vez de 1.900, y sin descargar ni
un registro individual.

Cada consulta lleva estos filtros fijos:

```
country=CL                  solo Chile
hasCoordinate=true          descarta registros sin coordenada
hasGeospatialIssue=false    descarta coordenadas marcadas como erróneas por GBIF
mediaType=StillImage        SOLO registros con fotografía
```

El último filtro es el que convierte esto en un dataset de imágenes: cada registro
contado tiene una foto real asociada.

In [14]:
from core.ingesta import ejecutar_ingesta

# Los CSV procesados quedan en caché. Poner True para volver a consultar GBIF (~3 min).
FORZAR_DESCARGA = False

rutas = {n: DIR_PROC / f"{n}.csv"
         for n in ["zonas_especies", "zonas_resumen", "fotos_muestra"]}

if FORZAR_DESCARGA or not all(r.exists() for r in rutas.values()):
    print("Consultando GBIF, esto toma unos minutos…")
    ejecutar_ingesta()

zonas_especies = pd.read_csv(rutas["zonas_especies"])
zonas_resumen = pd.read_csv(rutas["zonas_resumen"])
fotos = pd.read_csv(rutas["fotos_muestra"])

print(f"zonas_especies : {len(zonas_especies):>6} filas  (una por zona × especie encontrada)")
print(f"zonas_resumen  : {len(zonas_resumen):>6} filas  (una por zona urbana)")
print(f"fotos_muestra  : {len(fotos):>6} filas  (registros con URL de fotografía)")
zonas_especies.head()

zonas_especies :   2543 filas  (una por zona × especie encontrada)
zonas_resumen  :     30 filas  (una por zona urbana)
fotos_muestra  :    180 filas  (registros con URL de fotografía)


,zona,speciesKey,registros
0,Gran Santiago,1340503,386
1,Gran Santiago,3117424,92
2,Gran Santiago,3190653,86
3,Gran Santiago,2979474,69
4,Gran Santiago,5371743,31


### El denominador: por qué necesitamos el total de registros de cada zona

Este es el punto metodológico más importante de todo el proyecto.

Si comparamos ciudades por **número de registros invasores**, Santiago gana siempre — pero
no porque esté más invadido, sino porque tiene millones de habitantes y miles de personas
usando iNaturalist. Estaríamos midiendo **cuánta gente saca fotos**, no cuánta invasión hay.

Por eso `zonas_resumen` incluye `registros_totales`: **todos** los registros fotográficos de
esa zona, de cualquier especie, nativa o exótica. La métrica que usamos es entonces:

$$\text{presión de invasión} = \frac{\text{registros de especies exóticas}}{\text{registros totales de la zona}}$$

Como el esfuerzo de observación afecta al numerador y al denominador por igual, **se
cancela**. Una proporción sí es comparable entre ciudades; un conteo bruto no.

In [15]:
zonas_resumen[["zona", "region", "radio_km", "registros_totales"]] \
    .sort_values("registros_totales", ascending=False).head(10)

,zona,region,radio_km,registros_totales
0,Gran Santiago,Metropolitana,30,48832
2,Gran Concepcion,Biobio,20,30798
1,Gran Valparaiso,Valparaiso,18,20519
3,Gran La Serena,Coquimbo,15,19609
14,Valdivia,Los Rios,12,8698
21,San Antonio,Valparaiso,12,8407
5,Temuco,Araucania,12,4368
18,Punta Arenas,Magallanes,12,4249
11,Puerto Montt,Los Lagos,15,3408
9,Arica,Arica y Parinacota,12,2794


---
# Paso 6 — Limpieza (parte 3): el problema del tamaño de muestra

Las zonas chicas tienen pocos registros, y una proporción calculada sobre pocos datos es
inestable: si una zona tiene 40 registros y 15 son exóticos, el 37,5% resultante puede
moverse muchísimo con un puñado de observaciones más.

Fijamos un **umbral mínimo de 500 registros totales** por zona para incluirla en los
rankings. Es una decisión arbitraria pero necesaria, y la dejamos explícita: las zonas
excluidas se reportan, no se esconden.

In [16]:
UMBRAL_MINIMO = 500

excluidas = zonas_resumen[zonas_resumen.registros_totales < UMBRAL_MINIMO]
incluidas = zonas_resumen[zonas_resumen.registros_totales >= UMBRAL_MINIMO]

print(f"Zonas incluidas en el análisis : {len(incluidas)}")
print(f"Zonas excluidas por pocos datos: {len(excluidas)}")
if len(excluidas):
    print()
    print(excluidas[["zona", "region", "registros_totales"]].to_string(index=False))
    print("\nEstas zonas quedan fuera de los rankings pero se conservan en los datos.")

Zonas incluidas en el análisis : 25
Zonas excluidas por pocos datos: 5

       zona      region  registros_totales
     Calama Antofagasta                321
    Copiapo     Atacama                425
Los Angeles      Biobio                465
     Ovalle    Coquimbo                456
    Linares       Maule                215

Estas zonas quedan fuera de los rankings pero se conservan en los datos.


### Un sesgo que NO podemos eliminar, y hay que declararlo

La proporción controla el esfuerzo de observación, pero queda un sesgo de fondo:
**las especies exóticas son más fáciles de fotografiar que las nativas.** Viven donde
está la gente, muchas son grandes y llamativas (una cotorra verde chillona contra un
insecto nativo de 3 mm), y varias son familiares para quien viene de Europa.

Eso significa que nuestras proporciones probablemente **sobreestiman** la presencia real
de exóticas. Lo que sí es válido es la **comparación entre ciudades**: el sesgo opera de
forma parecida en todas, así que el *ranking* se sostiene incluso si los valores absolutos
están inflados.

Es una limitación honesta del dato disponible, no un error corregible.

---
# Paso 7 — La dimensión fotográfica

Como cada registro tiene una imagen, guardamos también una muestra con las URLs. Esto
sirve para dos cosas: ilustrar el catálogo de la aplicación con fotos reales tomadas en
Chile, y ser el punto de partida si más adelante entrenamos el clasificador.

In [17]:
print("Licencias de las fotografías de la muestra:")
print(fotos.licencia.value_counts().to_string())
print()
print(f"Registros con URL de foto disponible: {fotos.foto_url.notna().sum()} de {len(fotos)}")
fotos[["nombre_gbif", "anio", "region_gbif", "licencia", "foto_url"]].head(5)

Licencias de las fotografías de la muestra:
licencia
http://creativecommons.org/licenses/by-nc/4.0/legalcode       149
http://creativecommons.org/licenses/by/4.0/legalcode           24
http://creativecommons.org/publicdomain/zero/1.0/legalcode      7

Registros con URL de foto disponible: 180 de 180


,nombre_gbif,anio,region_gbif,licencia,foto_url
0,Bombus terrestris,2026,Bío-Bío,http://creativecommons.org/licenses/by-nc/4.0/...,https://inaturalist-open-data.s3.amazonaws.com...
1,Bombus terrestris,2026,Magallanes y Antártica Chilena,http://creativecommons.org/licenses/by-nc/4.0/...,https://inaturalist-open-data.s3.amazonaws.com...
2,Bombus terrestris,2026,Libertador General Bernardo O'Higgins,http://creativecommons.org/licenses/by-nc/4.0/...,https://inaturalist-open-data.s3.amazonaws.com...
3,Harmonia axyridis,2026,Los Lagos,http://creativecommons.org/licenses/by-nc/4.0/...,https://inaturalist-open-data.s3.amazonaws.com...
4,Harmonia axyridis,2026,Bío-Bío,http://creativecommons.org/licenses/by-nc/4.0/...,https://inaturalist-open-data.s3.amazonaws.com...


**Nota sobre licencias y uso de las imágenes.** La mayoría es CC-BY-NC 4.0: se pueden usar
citando al autor, **sin fines comerciales**. Para este proyecto académico está bien, pero
hay que citar la fuente en la app. Si algún día se usara comercialmente, habría que
filtrar solo las CC0 y CC-BY.

También aparece aquí el desorden de `stateProvince` en GBIF, que es texto libre escrito
por cada observador: conviven `"Región del Bíobío"` y `"Bío-Bío"` para la misma región, y
hay códigos numéricos sin normalizar. **Nosotros no usamos ese campo** — asignamos la zona
por distancia a las coordenadas, que es un dato numérico y confiable. Vale la pena
mostrarlo porque es la trampa evidente en la que caería un análisis que confiara en el
nombre de la región.

In [18]:
print("Ejemplos del desorden en stateProvince (campo de texto libre de GBIF):")
print(fotos.region_gbif.dropna().value_counts().head(12).to_string())

Ejemplos del desorden en stateProvince (campo de texto libre de GBIF):
region_gbif
Región Metropolitana de Santiago             47
Valparaíso                                   37
Bío-Bío                                      25
Los Lagos                                    18
Magallanes y Antártica Chilena               15
Libertador General Bernardo O'Higgins        11
Coquimbo                                      8
Los Ríos                                      8
Araucanía                                     5
Maule                                         4
Tarapacá                                      1
Aisén del General Carlos Ibáñez del Campo     1


---
# Resumen del paso de obtención y limpieza

| # | Problema encontrado | Cómo lo resolvimos |
|---|---|---|
| 1 | GRIIS no tiene región ni fecha | Cruzamos con GBIF, que sí tiene coordenadas |
| 2 | Nombres con autoría (`"Acacia dealbata Link"`) | Extrajimos el binomio género + especie |
| 3 | Sinónimos antiguos (`Helix aspersa`) | Emparejamos por `speciesKey`, no por texto |
| 4 | `isInvasive` vacío en 598 filas | Separamos "exótica" de "invasora declarada" |
| 5 | Conurbaciones se contarían dos veces | Unimos Gran Valparaíso, Concepción y La Serena |
| 6 | Ciudades con distinto tamaño | Radio proporcional, verificado sin solapes |
| 7 | Más observadores ≠ más invasión | Normalizamos por registros totales de la zona |
| 8 | Zonas con muy pocos datos | Umbral de 500 registros, exclusiones reportadas |
| 9 | `stateProvince` sucio en GBIF | No lo usamos: asignamos zona por coordenadas |

**Archivos generados en `data/procesado/`:**

- `catalogo_especies.csv` — 812 especies exóticas con su `speciesKey` de GBIF
- `zonas_especies.csv` — registros fotográficos por zona urbana × especie
- `zonas_resumen.csv` — 30 zonas con su total de registros (el denominador)
- `fotos_muestra.csv` — muestra de registros con URL de imagen y licencia

El análisis continúa en [`02_analisis_urbano.ipynb`](02_analisis_urbano.ipynb).